### Links and Resources
| Function | Use |
|---|---|
| [`Window`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Window.html) | Define partition / order / frame |
| [`row_number`, `rank`, `dense_rank`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.row_number.html) | Position within a partition |
| [`lag`, `lead`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.lag.html) | Look back / forward |
| [`percent_rank`, `cume_dist`, `ntile`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.percent_rank.html) | Quantile-like positions |
| [Window function intro](https://www.databricks.com/blog/2015/07/15/introducing-window-functions-in-spark-sql.html) | Background |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel \u2014 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

from pyspark.sql import functions as F


def display(df, n=20):
    df.show(n, truncate=False)

# Window Functions

A window function computes a value **for each row** based on a **window of related rows** — without collapsing rows the way `groupBy` does. They answer questions like:

- "What's each employee's rank within their department?"
- "What's the running 7-day total of sales per store?"
- "What was the previous order date for this customer?"

A window has three parts:

1. **`partitionBy`** — group the rows (e.g. per department).
2. **`orderBy`** — order within the partition (often required).
3. **Frame** — `rowsBetween(...)` or `rangeBetween(...)` — which rows around the current row are visible. Defaults vary by function.

In [ ]:
sales = spark.createDataFrame([
    ("ACME", "2024-01-01", 100),
    ("ACME", "2024-01-02", 150),
    ("ACME", "2024-01-03", 120),
    ("ACME", "2024-01-04", 200),
    ("ACME", "2024-01-05",  80),
    ("BETA", "2024-01-01", 300),
    ("BETA", "2024-01-02", 250),
    ("BETA", "2024-01-03", 400),
    ("BETA", "2024-01-04", 350),
], ["store", "date", "sales"]).withColumn("date", F.to_date("date"))

display(sales)

## `Window.partitionBy(...).orderBy(...)`

In [ ]:
from pyspark.sql.window import Window

w = Window.partitionBy("store").orderBy("date")

sales.select(
    "store", "date", "sales",
    F.row_number().over(w).alias("row_num"),
    F.rank().over(w).alias("rank"),
    F.dense_rank().over(w).alias("dense_rank"),
).show()

### `row_number` vs `rank` vs `dense_rank`

| Function | Ties | Gaps after ties |
|---|---|---|
| `row_number` | Broken arbitrarily — every row gets a unique number | n/a |
| `rank` | Same value for ties | Yes — next rank skips |
| `dense_rank` | Same value for ties | No |

Use `row_number` to pick "the one row per group" (e.g. latest record per customer). Use `rank` / `dense_rank` for leaderboards.

## Pick the latest row per group — the canonical window pattern

This pattern shows up in every project. Memorize it.

In [ ]:
latest_per_store = (sales
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("store").orderBy(F.col("date").desc())
    ))
    .filter(F.col("rn") == 1)
    .drop("rn"))

display(latest_per_store)

## `lag` and `lead` — look back / forward

`lag(col, n=1)` returns the value from `n` rows earlier in the partition. `lead` looks forward. Both return NULL when no such row exists.

In [ ]:
sales.select(
    "store", "date", "sales",
    F.lag("sales", 1).over(w).alias("prev_sales"),
    (F.col("sales") - F.lag("sales", 1).over(w)).alias("delta"),
    F.lead("sales", 1).over(w).alias("next_sales"),
).show()

## Aggregations over a window — running totals & moving averages

When you put `sum`, `avg`, `count`, etc. `.over(window)`, you get a per-row aggregate computed over the window's frame.

**Important: the default frame depends on whether you ordered.**

- **No `orderBy`** → unbounded frame (whole partition).
- **With `orderBy`** → `rangeBetween(unboundedPreceding, currentRow)` — i.e. running aggregate.

To get something else (rolling window of N rows, centered window, etc.), set the frame explicitly.

In [ ]:
# Running total per store
sales.select(
    "store", "date", "sales",
    F.sum("sales").over(w).alias("running_total"),
    F.avg("sales").over(w).alias("running_avg"),
).show()

## Frames — `rowsBetween` vs `rangeBetween`

- **`rowsBetween(a, b)`** — physical row offsets. `rowsBetween(-2, 0)` = current row plus the two preceding rows.
- **`rangeBetween(a, b)`** — logical offsets on the *value* of the order column. `rangeBetween(-7, 0)` over a date column = last 7 calendar days.

Constants: `Window.unboundedPreceding`, `Window.unboundedFollowing`, `Window.currentRow`.

In [ ]:
rolling3 = w.rowsBetween(-2, 0)        # 3-row window: prev 2 + current
sales.select(
    "store", "date", "sales",
    F.avg("sales").over(rolling3).alias("avg_3row"),
    F.max("sales").over(rolling3).alias("max_3row"),
).show()

In [ ]:
# 7-day rolling sum using rangeBetween on a date column.
# Spark needs the order column expressed as a number for rangeBetween — cast to long days since epoch.
days = F.col("date").cast("long") / (24 * 3600)

w_days = (Window.partitionBy("store")
    .orderBy(days)
    .rangeBetween(-6, 0))                 # last 6 days + today = 7 calendar days

sales.select(
    "store", "date", "sales",
    F.sum("sales").over(w_days).alias("rolling_7d_sum"),
).show()

## Distribution — `percent_rank`, `cume_dist`, `ntile`

- `percent_rank` — `(rank - 1) / (n - 1)`, ranges 0..1.
- `cume_dist` — fraction of rows with value ≤ current.
- `ntile(k)` — split partition into `k` roughly-equal buckets.

In [ ]:
sales.select(
    "store", "date", "sales",
    F.percent_rank().over(w).alias("pct_rank"),
    F.cume_dist().over(w).alias("cume"),
    F.ntile(3).over(w).alias("tertile"),
).show()

## SQL form — same thing with `OVER (...)`

In [ ]:
sales.createOrReplaceTempView("sales")

spark.sql("""
SELECT store, date, sales,
       row_number()                        OVER (PARTITION BY store ORDER BY date)        AS rn,
       sales - LAG(sales) OVER (PARTITION BY store ORDER BY date)                         AS delta,
       AVG(sales) OVER (PARTITION BY store ORDER BY date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS avg_3row
FROM sales
""").show()

## Performance notes

- Each distinct window spec triggers a **shuffle and sort** by the partition keys. Reuse the same `Window` object when columns share a spec.
- Skewed partition keys (one store with millions of rows) hurt — same problem as a skewed `groupBy`. Salting helps if it's a hot spot.
- `row_number` + filter (`rn == 1`) is often beaten by a `groupBy(...).agg(max_struct(...))` trick when you only need a few columns.

## Takeaways

- Three knobs: **partition** (which rows are siblings), **order** (their sequence), **frame** (which siblings are visible).
- `row_number` for "pick one per group", `rank/dense_rank` for leaderboards.
- `lag/lead` for previous/next row deltas.
- Running aggregate is the default when you `orderBy`; set `rowsBetween`/`rangeBetween` for anything else.
- Reuse `Window` objects to avoid duplicate shuffles.